In [ ]:
import os
import json
import warnings
import numpy as np
import xarray as xr
import proplot as pplt
warnings.filterwarnings('ignore')
pplt.rc.update({
    'savefig.dpi':900,
    'savefig.bbox':'tight',
    'savefig.pad_inches':0.02,
    'tick.minor':False,
    'font.size':9,
    'label.size':9,
    'tick.labelsize':9,
    'title.size':9,
    'abc.size':9,
    'legend.fontsize':9,
    'suptitle.size':9,
    'leftlabelsize':9,
    'toplabelsize':9,
    'leftlabel.weight':'normal',
    'toplabel.weight':'normal',
    'reso':'xx-hi'})

In [ ]:
with open('../scripts/configs.json','r',encoding='utf-8') as f:
    CONFIGS = json.load(f)
SPLITSDIR  = CONFIGS['filepaths']['splits']
PREDSDIR   = CONFIGS['filepaths']['predictions']
LATRANGE   = CONFIGS['domain']['latrange']
LONRANGE   = CONFIGS['domain']['lonrange']
TARGETVAR  = CONFIGS['domain']['target']
SPLIT      = 'test'
NBINS      = 100

NNMODELS = {}
SRMODELS = {}
for name,rc in CONFIGS['experiments']['nn']['runs'].items():
    predpath = os.path.join(PREDSDIR,f'{name}_{SPLIT}_predictions.nc')
    if not os.path.exists(predpath):
        continue
    NNMODELS[name] = {'label':rc['description'],'color':rc['color']}
for name,rc in CONFIGS['experiments']['sr']['optimizedeqs'].items():
    predpath = os.path.join(PREDSDIR,f'{name}_{SPLIT}_predictions.nc')
    if not os.path.exists(predpath):
        continue
    SRMODELS[name] = {'label':rc['description'],'color':rc['color']}
ALLMODELS = {**NNMODELS,**SRMODELS}
ORDER     = list(ALLMODELS.keys())
print(f'Found {len(NNMODELS)} NN and {len(SRMODELS)} SR models')

In [ ]:
def hellinger(p,q,nbins=100,range=None):
    phist,edges = np.histogram(p,bins=nbins,range=range,density=True)
    qhist,_     = np.histogram(q,bins=edges,density=True)
    bw = edges[1]-edges[0]
    return np.sqrt(1-np.sum(np.sqrt(phist*qhist)*bw))

In [ ]:
with xr.open_dataset(os.path.join(SPLITSDIR,f'{SPLIT}.h5'),engine='h5netcdf') as ds:
    obsda = ds[TARGETVAR].load()
    lf    = ds['lf'].values if 'time' not in ds['lf'].dims else ds['lf'].isel(time=0).values

ntime,nlat,nlon = obsda.sizes['time'],obsda.sizes['lat'],obsda.sizes['lon']
lat,lon = obsda.lat.values,obsda.lon.values
obsflat = obsda.values.ravel()
lfflat  = np.tile(lf,(ntime,1,1)).ravel() if lf.ndim==2 else lf.ravel()

preds = {}
for name in ORDER:
    filepath = os.path.join(PREDSDIR,f'{name}_{SPLIT}_predictions.nc')
    with xr.open_dataset(filepath) as ds:
        pred = ds[TARGETVAR].load()
    if 'seed' in pred.dims:
        pred = pred.mean('seed')
    if 'complexity' in pred.dims:
        pred = pred.isel(complexity=0)
    preds[name] = pred.reindex(lat=lat,lon=lon,method='nearest').transpose('time','lat','lon').values.ravel()

valid = np.isfinite(obsflat)
for p in preds.values():
    valid &= np.isfinite(p)
print(f'Loaded {len(preds)} models, {valid.sum():,} valid samples')

In [ ]:
obsvalid = obsflat[valid]
histrange = (0,np.percentile(obsvalid,99.9))

print(f'{"Model":<16} {"H (all)":>10} {"H (>0)":>10}')
print('-'*38)
for name in ORDER:
    predvalid = preds[name][valid]
    hall = hellinger(obsvalid,predvalid,nbins=NBINS,range=histrange)
    posmask  = (obsvalid>0)|(predvalid>0)
    hwet = hellinger(obsvalid[posmask],predvalid[posmask],nbins=NBINS,range=histrange)
    print(f'{ALLMODELS[name]["label"]:<16} {hall:10.4f} {hwet:10.4f}')

In [ ]:
landmask  = valid & (lfflat>0.5)
oceanmask = valid & (lfflat<0.5)

print(f'{"Model":<16} {"H (land)":>10} {"H (ocean)":>10}')
print('-'*38)
for name in ORDER:
    hland  = hellinger(obsflat[landmask],preds[name][landmask],nbins=NBINS,range=histrange)
    hocean = hellinger(obsflat[oceanmask],preds[name][oceanmask],nbins=NBINS,range=histrange)
    print(f'{ALLMODELS[name]["label"]:<16} {hland:10.4f} {hocean:10.4f}')

In [ ]:
hmap = np.full((len(ORDER),nlat,nlon),np.nan)
obsreshaped = obsda.values
for i,name in enumerate(ORDER):
    predreshaped = preds[name].reshape(ntime,nlat,nlon)
    for j in range(nlat):
        for k in range(nlon):
            o = obsreshaped[:,j,k]
            p = predreshaped[:,j,k]
            m = np.isfinite(o)&np.isfinite(p)
            if m.sum()<50:
                continue
            hmap[i,j,k] = hellinger(o[m],p[m],nbins=50,range=histrange)

ncols = min(len(ORDER),4)
nrows = (len(ORDER)+ncols-1)//ncols
fig,axs = pplt.subplots(nrows=nrows,ncols=ncols,proj='cyl',figwidth=10,share=True)
axs.format(coast=True,latlim=LATRANGE,lonlim=LONRANGE,latlines=[10,15,20],lonlines=[65,75,85],grid=False)
axs[-1,:].format(lonlabels='b')
axs[:,0].format(latlabels='l')
m = None
for idx,(ax,name) in enumerate(zip(axs,ORDER)):
    m = ax.pcolormesh(lon,lat,hmap[idx],cmap='Reds',vmin=0,vmax=0.5,levels=11,extend='max')
    ax.format(title=ALLMODELS[name]['label'])
for ax in axs[len(ORDER):]:
    ax.set_visible(False)
fig.colorbar(m,loc='b',label='Hellinger Distance')
axs.format(abc=True,titleloc='l')
pplt.show()

In [ ]:
hvalues = [hellinger(obsflat[valid],preds[name][valid],nbins=NBINS,range=histrange) for name in ORDER]

fig,ax = pplt.subplots(figwidth=4,refheight=2.5)
ax.bar(np.arange(len(ORDER)),hvalues,color=[ALLMODELS[name]['color'] for name in ORDER],alpha=0.85)
ax.format(grid=False,xticks=np.arange(len(ORDER)),xticklabels=[ALLMODELS[name]['label'] for name in ORDER],
          xrotation=45,ylabel='Hellinger Distance',ylim=(0,max(hvalues)*1.2),
          title='Precipitation Distribution Similarity')
pplt.show()